Esercizio 1
Input: un file transactions.csv con colonne date, product_id, qty, price.
Compito: leggere il CSV (parse_dates=["date"]), assicurarti che product_id sia category, calcolare revenue= qty*price e stampare le 5 righe con revenue più alte.

In [5]:
import numpy as np
import pandas as pd 
from pathlib import Path

n_rows = 100
np.random.seed(123)

# crezione dati in maniera casuale 

start_date = np.datetime64("2023-01-01") # salvato in datetime64 (numero interdo di giorni dal 1-1-1970)
end_date = np.datetime64("2024-12-31")
date_diff = (end_date - start_date).astype(int) # differenza di giorni
random_days = np.random.randint(0, date_diff, size=n_rows)
dates = start_date + random_days # somma vettoriale di numpy 

product_id = np.arange(1, n_rows + 1) # id_prodotto 

qty = np.random.randint(1, 100, size= n_rows)

endings = np.array([0.99, 0.95, 0.50, 0.00]) # fine prezzo
weights = np.array([0.50, 0.25, 0.15, 0.10]) # probabilità fine prezzo 
random_endings = np.random.choice(endings, size= n_rows, p=weights)
unit_base_price = np.random.randint(1, 1000, size = n_rows)
unit_price = unit_base_price + random_endings # prezzo unitario completo 

df_transactions = pd.DataFrame({"product_id":product_id, "date":dates, "unit_price":unit_price,"quantity":qty})

# per far creare il file nella stessa cartella dello script 
try:
    cartella_script = Path(__file__)
except NameError:
    cartella_script = Path.cwd()

percorso_file = cartella_script / 'transactions.csv'
df_transactions.to_csv(percorso_file, index= False)


# leggiamo il file csv salvato
df_transactions_csv = pd.read_csv(percorso_file, parse_dates=["date"])
df_transactions_csv["revenue"] = df_transactions_csv["quantity"] * df_transactions_csv["unit_price"]

df_ordinato = df_transactions_csv.sort_values(by="unit_price", ascending=False)
print(df_ordinato.head())









    product_id       date  unit_price  quantity   revenue
24          25 2023-07-28      995.95         8   7967.60
97          98 2023-07-18      992.99        24  23831.76
3            4 2023-11-19      987.99        62  61255.38
30          31 2023-11-19      987.50        45  44437.50
34          35 2023-06-26      979.99        71  69579.29


Esercizio 2 (intermedio)
Input: sales_*.csv ( piu file mensili), stores.jsonl (store meta), tabella promotions in SQLite.
Compito: leggere tutti i CSV in straming (chunksize), unirli ai metadata JSON e alla tabella SQL, ottimizzare tipi, calcolare vendite mensili per regione.

In [34]:
import numpy as np
import pandas as pd 
from pathlib import Path 
import sqlite3
import json 

# per far creare il file nella stessa cartella dello script 
try:
    cartella_script = Path(__file__)
except NameError:
    cartella_script = Path.cwd()

n_rows = 100
np.random.seed(123)

prodotti = ["Laptop", "Mouse", "Tastiera", "Monitor", "Cuffie"]

prezzi_dict = {
    "Laptop": 899.99,
    "Mouse": 19.99,
    "Tastiera": 49.99,
    "Monitor": 199.99,
    "Cuffie": 29.99,
}
# Date di inizio e fine per ciascun mese (Gennaio, Febbraio, Marzo 2024)
mesi_info = [
    (1, "2025-01-01", "2025-01-31"),
    (2, "2025-02-01", "2025-02-28"), 
    (3, "2025-03-01", "2025-03-31"),
]

# per i le vendite CSV
stores = ["STORE_001","STORE_002","STORE_003"]

for mese_num, start_str, end_str in mesi_info:
    # generazione date casuali per il mese corrente
    start_date = np.datetime64(start_str)
    end_date = np.datetime64(end_str)
    date_diff = (end_date - start_date).astype(int) + 1
    random_days = np.random.randint(0, date_diff, size = n_rows)
    dates = start_date + random_days

    # generazione prodotti casuali 
    prodotto_sample = np.random.choice(prodotti, size= n_rows) # qua crea tutti i 100 prodotti casualmente

    # generazione store casuale
    quantita_sample = np.random.randint(1,10, size=n_rows)

    # generazione store casuale
    store_sample = np.random.choice(stores, size=n_rows)

    #generazione prezzi casuale 
    unit_price_sample = np.array([prezzi_dict[p] for p in prodotto_sample])
    total_sample = np.round(unit_price_sample * quantita_sample, 2)

    # creazione dataframe per poi salvare in csv 
    df_sales = pd.DataFrame(
        {
            "id_transazione": np.arange(1, n_rows + 1),
            "id_store": store_sample,
            "data": dates,
            "prodotto": prodotto_sample,
            "quantita": quantita_sample,
            "unite_price": unit_price_sample,
            "totale": total_sample,
        }
    )

    # nominazione file e percorso file CSV
    filename = f"sales_{mese_num}.csv"
    percorso_file = cartella_script / filename
    df_sales.to_csv(percorso_file, index=False)
    print(f"File {percorso_file} creato con successo!")



# creazion file json per gli store

stores_data = [
    {
        "id_store": "STORE_001",
        "nome_store": "Milano Duomo",
        "citta": "Milano",
        "regione": "Lombardia",
        "superficie_mq": 450,
        "attivo": True,
    },
    {
        "id_store": "STORE_002",
        "nome_store": "Roma Centro",
        "citta": "Roma",
        "regione": "Lazio",
        "superficie_mq": 600,
        "attivo": True,
    },
    {
        "id_store": "STORE_003",
        "nome_store": "Torino Via Roma",
        "citta": "Torino",
        "regione": "Piemonte",
        "superficie_mq": 320,
        "attivo": True,
    },
]

df_store = pd.DataFrame(stores_data)
filename = f"stores.jsonl"
percorso_file = cartella_script / filename

# esportazione in formato JSON Lines 
df_store.to_json(percorso_file, orient="records",lines=True)
print(f"File {filename} creato con successo!")


# creazione connessione database in SQLite
percorso_db = cartella_script / "database_vendite.db"
conn = sqlite3.connect(percorso_db) # qui inserisco il percorso

# Creazione delle promozioni specifiche per ogni combinazione Store + Prodotto
promozioni_list = [
    # Store 001 - Milano
    {
        "store_id": "STORE_001",
        "prodotto": "Laptop",
        "tipo_sconto": "PERCENTUALE",
        "valore_sconto": 10.0,
    },
    {
        "store_id": "STORE_001",
        "prodotto": "Mouse",
        "tipo_sconto": "FISSO",
        "valore_sconto": 5.0,
    },
    {
        "store_id": "STORE_001",
        "prodotto": "Tastiera",
        "tipo_sconto": "PERCENTUALE",
        "valore_sconto": 15.0,
    },
    {
        "store_id": "STORE_001",
        "prodotto": "Monitor",
        "tipo_sconto": "FISSO",
        "valore_sconto": 20.0,
    },
    {
        "store_id": "STORE_001",
        "prodotto": "Cuffie",
        "tipo_sconto": "PERCENTUALE",
        "valore_sconto": 5.0,
    },
    # Store 002 - Roma
    {
        "store_id": "STORE_002",
        "prodotto": "Laptop",
        "tipo_sconto": "PERCENTUALE",
        "valore_sconto": 12.0,
    },
    {
        "store_id": "STORE_002",
        "prodotto": "Mouse",
        "tipo_sconto": "PERCENTUALE",
        "valore_sconto": 20.0,
    },
    {
        "store_id": "STORE_002",
        "prodotto": "Tastiera",
        "tipo_sconto": "FISSO",
        "valore_sconto": 10.0,
    },
    {
        "store_id": "STORE_002",
        "prodotto": "Monitor",
        "tipo_sconto": "PERCENTUALE",
        "valore_sconto": 15.0,
    },
    {
        "store_id": "STORE_002",
        "prodotto": "Cuffie",
        "tipo_sconto": "FISSO",
        "valore_sconto": 7.0,
    },
    # Store 003 - Torino
    {
        "store_id": "STORE_003",
        "prodotto": "Laptop",
        "tipo_sconto": "FISSO",
        "valore_sconto": 100.0,
    },
    {
        "store_id": "STORE_003",
        "prodotto": "Mouse",
        "tipo_sconto": "FISSO",
        "valore_sconto": 3.0,
    },
    {
        "store_id": "STORE_003",
        "prodotto": "Tastiera",
        "tipo_sconto": "PERCENTUALE",
        "valore_sconto": 10.0,
    },
    {
        "store_id": "STORE_003",
        "prodotto": "Monitor",
        "tipo_sconto": "FISSO",
        "valore_sconto": 30.0,
    },
    {
        "store_id": "STORE_003",
        "prodotto": "Cuffie",
        "tipo_sconto": "PERCENTUALE",
        "valore_sconto": 25.0,
    },
]

# creazione dataframe promotions
df_promotions = pd.DataFrame(promozioni_list)

# salvataggio tabella in sql lite
df_promotions.to_sql("promotions", conn, if_exists="replace",index=False)
print("Tabella creata con successo!")
print(df_promotions.head())

conn.close() # ricordare sempre di chiudere la connessione

##################################################################################

# SECONDA FASE DELL'ESERCIZIO

##################################################################################
import glob 

# 1. recupera la lista di tutti i file csv delle vendite 
file_csv = list(cartella_script.glob("sales_*.csv"))
chunk_sales_list_csv = [] 

# definizione dimensione del blocco ( 20 righe alla volta )
CHUNK_SIZE = 20

# 2. iterazione su ogni file csv 
for file in file_csv: 
    print(f"\n--- Inizio lettura in streaming di: {file}")
    # read_csv restituisce un iteratore
    for chunk in pd.read_csv(file, parse_dates=["data"], chunksize= CHUNK_SIZE):
        chunk_sales_list_csv.append(chunk)

# contatenazione di tutti i blocchi in un UNICO dataframe sales
df_total_sales = pd.concat(chunk_sales_list_csv, ignore_index=True)
print(df_total_sales.head())


##################################################################################

# TERZA FASE DELL'ESERCIZIO

##################################################################################


# 1. leggere la tabella SWL in un DataFrame Pandas
conn = sqlite3.connect(percorso_db) # qui inserisco il percorso
df_promotions = pd.read_sql_query("SELECT * FROM promotions", conn)

# 2. carica i metadata dello store dal file JSON Lines
filename = f"stores.jsonl"
percorso_file = cartella_script / filename
df_stores = pd.read_json(percorso_file, orient ="records", lines=True)

# 3. UNIONE 1: sales + stores (JOIN su id_store)
# usiamo left per conservare tutte le vendite anche se mancassero metadata
sales_with_stores = pd.merge(df_total_sales, df_stores, on="id_store", how="left")

# 4. UNIONE 2: sales_with_stores + promotions (JOIN su id_store e prodotto)
# Nota: se nella colonna sales hai prodotto e nella tabella promozioni la colonna ha lo stesso nome
# specificati i campi sia left_on che right_on perchè non si chiamano esattamente allo stesso modo
df_completo = pd.merge(
    sales_with_stores,
    df_promotions,
    left_on=["id_store","prodotto"],
    right_on=["store_id","prodotto"],
    how="left"
)

conn.close() # seconda chiusura DB

print("--- Dataframe Unificato Completo ---")
filename = "database_vendite.csv"
percorso_file = cartella_script / filename
df_completo.to_csv("database_vendite.csv", index=False)
print(f"Totale righe: {len(df_completo)}")
df_completo["data"] = pd.to_datetime(df_completo["data"])

print("######################## DATAFRAME ################################")
print(df_completo.sort_values(by="data", ascending=False))


##################################################################################

# QUARTA FASE DELL'ESERCIZIO

##################################################################################

# 1. ottimizzazione dei tipi di dato (ottimizzazione memoria)
# convertiamo le stringhe ripetitive in category e riduciamo l'occupazione della memoria
df_completo["id_store"] = df_completo["id_store"].astype("category")
df_completo["prodotto"] = df_completo["prodotto"].astype("category")
df_completo["citta"] = df_completo["citta"].astype("category")
df_completo["regione"] = df_completo["regione"].astype("category")
df_completo["attivo"] = df_completo["attivo"].astype("category")
df_completo["tipo_sconto"] = df_completo["tipo_sconto"].astype("category")

# Riduzione dei tipi numerici
df_completo["quantita"] = df_completo["quantita"].astype("int16")
df_completo["superficie_mq"] = df_completo["superficie_mq"].astype("int16")
df_completo["unite_price"] = df_completo["unite_price"].astype("float32")
df_completo["totale"] = df_completo["totale"].astype("float32")
df_completo["valore_sconto"] = df_completo["valore_sconto"].astype("float32")

# rimoviamo la colonna duplicata store_id creata dal marge
if "store_id" in df_completo.columns:
    df_completo.drop(columns=["store_id"], inplace=True)

print("Tipi di dato ottimizzati:")
print(df_completo.dtypes)

# 2. CALCOLO DELL'IMPORTO SCONTATO EFFETTIVO
# calcoliamo il totale netto applicando le promozioni

def calcola_totale_scontato(row):
    totale = row["totale"]
    tipo = row["tipo_sconto"]
    valore = row["valore_sconto"]

    if pd.isna(tipo):
        return totale
    elif tipo =="PERCENTUALE":
        return np.round(totale * (1 - valore/100), 2)
    elif tipo == "FISSO":
        # proteggiamo il codice da prezzi negativi
        return np.round(max(0, totale - (valore * row["quantita"])),2)

# aggiungiamo la colonna totale_sconto
df_completo["totale_scontato"] = df_completo.apply(calcola_totale_scontato, axis=1).astype("float32")

# 3. CALCOLO VENDITE MENSILI PER REGIONE 
# estraiamo il mese/anno (formato YYYY-MM) dalla colonna data
df_completo["anno_mese"] = df_completo["data"].dt.to_period("M")

# Raggruppiamo per Anno-Mese e Regione
vendite_regionali_mensili = (
    df_completo.groupby(["anno_mese", "regione"], observed=False)
    .agg(
        totale_transazioni=("id_transazione","count"),
        unita_vendute=("quantita","sum"),
        fatturato_lordo=("totale","sum"),
        fatturato_netto=("totale_scontato","sum")
    ).reset_index()
)

# ordinamento per Mese e Fatturato netto decrescente
vendite_regionali_mensili.sort_values(
    by=["anno_mese","fatturato_netto"], ascending=[True,False], inplace=True
)

print("\n--- REPORT VENDITE MENSILI PER REGIONE ---")
print(vendite_regionali_mensili.to_string(index=False))



File c:\Users\raffy\OneDrive\Documenti\GitHub\Epicode_modulo2_python_per_data_science\09_creazione_avanzata_DataFrame_da_CSV_JSON_database_relazionali\sales_1.csv creato con successo!
File c:\Users\raffy\OneDrive\Documenti\GitHub\Epicode_modulo2_python_per_data_science\09_creazione_avanzata_DataFrame_da_CSV_JSON_database_relazionali\sales_2.csv creato con successo!
File c:\Users\raffy\OneDrive\Documenti\GitHub\Epicode_modulo2_python_per_data_science\09_creazione_avanzata_DataFrame_da_CSV_JSON_database_relazionali\sales_3.csv creato con successo!
File stores.jsonl creato con successo!
Tabella creata con successo!
    store_id  prodotto  tipo_sconto  valore_sconto
0  STORE_001    Laptop  PERCENTUALE           10.0
1  STORE_001     Mouse        FISSO            5.0
2  STORE_001  Tastiera  PERCENTUALE           15.0
3  STORE_001   Monitor        FISSO           20.0
4  STORE_001    Cuffie  PERCENTUALE            5.0

--- Inizio lettura in streaming di: c:\Users\raffy\OneDrive\Documenti\Git

Esercizio 3 (avanzato)
Input: dataset reale con: CSV grandi, JSON annidati clienti, DB SQL con prezzi e promozioni.
Compito: costruire una pipeline che: legge le sorgenti, normalizza JSON annidati, unisce tutto, applica best-pratice (usecols, dtype, parse_dates, downcast, category), identifica e gestisci valori mancanti, salva risultato pulito clean_data.csv e salva mapping categorie in file (pickle). Documenta ogni passaggio.


In [52]:
import pandas as pd 
import numpy as np
import json 
import sqlite3
from pathlib import Path 
import pickle

# per far creare il file nella stessa cartella dello script 
try:
    cartella_script = Path(__file__).resolve().parent
except NameError:
    cartella_script = Path.cwd() # in caso se __file__ non esistesse

np.random.seed(42)

n_rows = 500 # aumentiamo la dimensione 

# ---------------------------------------------------------
# 1. GENERAZIONE CSV VENDITE (Transazioni)
# ---------------------------------------------------------

prodotti = ["PROD_A","PROD_B","PROD_C","PROD_D"]
clienti_ids = [f"CLI_{i:04d}" for i in range(1,101)] # 100 clienti unici 

df_sales = pd.DataFrame({
    "id_transazione": np.arange(1000, 1000 + n_rows),
    "id_cliente": np.random.choice(clienti_ids, size=n_rows),
    "codice_prodotto":np.random.choice(prodotti, size=n_rows),
    "quantita": np.random.randint(1, 6, size=n_rows),
    "data_ora": pd.date_range(start="2025-01-01", periods=n_rows, freq="30min")
})

df_sales.to_csv(cartella_script/ "vendite_grandi.csv", index=False)
print("File vendite_grandi.csv creato.!")

# ---------------------------------------------------------
# 2. GENERAZIONE JSON ANNIDATO (Anagrafica Clienti)
# --------------------------------------------------------- 
# Struttura annidata: id -> info personali -> indirizzo (via, citta) -> preferenze

clienti_nested = []
citta_list = ["Milano","Roma","Torino","Napoli","Bologna"]

for cli_id in clienti_ids:
    clienti_nested.append(
        {
            "cliente_id": cli_id,
            "profilo": {
                "nome": f"Cliente_{cli_id}",
                "eta": int(np.random.randint(18,70)),
                "livello_fedelta": np.random.choice(["Bronze","Silver","Gold","Platinum"])
            },
            "residenza": {
                "citta":np.random.choice(citta_list),
                "cap": f"{np.random.randint(10000, 90000):05d}"
            },
            "contatti":{
                "email_verificata": bool(np.random.choice([True, False]))
            }
        },
    )

""" with open(cartella_script/"clienti_annidati.jsonl","w",encoding="utf-8") as f:
    json.dump(clienti_nested, f, indent=2)
print("File clienti_annidati.jsonl creato!") """

# Conversione lista di dizionari in DataFrame
df_clienti_temp = pd.DataFrame(clienti_nested)

# Esporta direttamente in JSON Lines
df_clienti_temp.to_json(
    cartella_script/"clienti_annidati.jsonl",
    orient="records",
    lines=True,
    force_ascii=True,
)


# ---------------------------------------------------------
# 3. GENERAZIONE DB SQLITE (Listino Prezzi e Sconti)
# ---------------------------------------------------------

conn = sqlite3.connect(cartella_script/"catalogo.db")
df_prezzi = pd.DataFrame({
    "codice_prodotto":prodotti,
    "categoria": ["Elettronica","Informatica","Elettronica","Accessori"],
    "prezzo_listino": [120.00, 450.00, 80.00, 25.00],
    "sconto_categoria_pct": [5.0, 10.0, 0.0, 15.0]
})

df_prezzi.to_sql("listino_prezzi", conn, if_exists="replace",index=False)
conn.close()
print("Database catalogo.db creato con successo.!")

# ==============================================================================
# 1. LETTURA E UNNESTING DEL JSON (Clienti)
# ==============================================================================

print("--- 1. Lettura e Appiattimento JSON Clienti ---")

# medoto di lettura file jsonl, lettura di ogni record con il for

data_json=[]

path_json = cartella_script / "clienti_annidati.jsonl"
with open(path_json,"r",encoding="utf-8") as f:
    for line in f:
        data_json.append(json.loads(line.strip()))

# pd.json_normalize converte i campi annidati (es. profilo.eta) in colonne piatte
df_clienti = pd.json_normalize(data_json)

#Pulizia nomi colonne: rinominiamo per comodità ed elimiano prefissi superflui
df_clienti.rename(
    columns={
        "cliente_id":"id_cliente",
        "profilo.nome":"nome_cliente",
        "profilo.eta":"eta",
        "profilo.livello_fedelta":"livello_fedelta",
        "residenza.citta":"citta_residenza",
        "residenza.cap":"cap",
        "contatti.email_verificata":"email_verificata",
    },
    inplace=True
)

# BEST PRACTICE ottimizzazione e downcasting per Clienti
df_clienti["id_cliente"] = df_clienti["id_cliente"].astype("category")
df_clienti["livello_fedelta"] = df_clienti["livello_fedelta"].astype("category")
df_clienti["citta_residenza"] = df_clienti["citta_residenza"].astype("category")
df_clienti["email_verificata"] = df_clienti["email_verificata"].astype("bool")

# downcasting numerico per l'età (int8 supporta valori da -128 a 127)
df_clienti["eta"] = pd.to_numeric(df_clienti["eta"], downcast="integer")

print(f"Clienti caricati: {len(df_clienti)} righe")
print(df_clienti[["id_cliente", "nome_cliente", "livello_fedelta", "citta_residenza"]].head(3))

# ==============================================================================
# 2. LETTURA CSV (Vendite)
# ==============================================================================

print("\n--- 2. Lettura CSV Vendite ---")

# [BEST PRACTICE] Definizione dei tipi in fase di lettura per ridurre l'allocazione RAM
dtypes_csv = {
    "id_transazione": "int32",
    "id_cliente": "category",
    "codice_prodotto": "category",
    "quantita": "int16",
}

# [BEST PRACTICE] Carichiamo solo le colonne strettamente necessarie
colonne_da_caricare = [
    "id_transazione",
    "id_cliente",
    "codice_prodotto",
    "quantita",
    "data_ora",
]

path_csv = cartella_script / "vendite_grandi.csv"

df_vendite = pd.read_csv(
    path_csv,
    usecols=colonne_da_caricare,
    dtype=dtypes_csv,
    parse_dates=["data_ora"], # parsing nativo e veloce delle date
)

print(f"Transazioni caricate: {len(df_vendite)} righe.")
print(df_vendite.head(3))

# ==============================================================================
# 3. LETTURA SQL (Catalogo Prezzi)
# ==============================================================================

print("\n--- 3. Lettura DB SQL Catalogo ---")

path_db = cartella_script / "catalogo.db"
conn = sqlite3.connect(path_db)

df_catalogo = pd.read_sql_query("SELECT * FROM listino_prezzi", conn)
conn.close()

# Ottimizzazione tipi catalogo 
df_catalogo["codice_prodotto"] = df_catalogo["codice_prodotto"].astype("category")
df_catalogo["categoria"] = df_catalogo["categoria"].astype("category")

# Downcasting float
df_catalogo["prezzo_listino"] = pd.to_numeric(
    df_catalogo["prezzo_listino"], downcast="float"
)
df_catalogo["sconto_categoria_pct"] = pd.to_numeric(
    df_catalogo["sconto_categoria_pct"], downcast="float"
)

print(f"Prodotti nel catalogo: {len(df_catalogo)} righe.")
print(df_catalogo)

# ==============================================================================
# 4. INTEGRAZIONE DEI DATASET (Merge / Join)
# ==============================================================================

print("\n--- 4. Unione del Dataset ---")

# Unione 1: Vendite + Clienti (JOIN su id_cliente)
df_merged = pd.merge(df_vendite, df_clienti, on="id_cliente", how="left")

# Unione 3: Catalogo prezzi (JOIN su codice_prodotto)
df_completo = pd.merge(df_merged, df_catalogo, on="codice_prodotto", how="left")

# --- IDENTIFICAZIONE VALORI MANCANTI ---
print("\nCheck valori mancanti per colonna:")
mancanti = df_completo.isnull().sum()
print(mancanti[mancanti > 0] if mancanti.sum() > 0 else "Nessun valore mancante rilevato!")

# --- GESTIONE DEI VALORI MANCANTI (Imputazione Strategica) ---
# 1. Impostiamo valori di default per stringhe/categorie
if "citta_residenza" in df_completo.columns:
    df_completo["citta_residenza"] = (
        df_completo["citta_residenza"].cat.add_categories(["Non Specificato"])
        
    )
    df_completo["citta_residenza"].fillna("Non Specificato")

if "livello_fedelta" in df_completo.columns:
    df_completo["livello_fedelta"] = (
        df_completo["livello_fedelta"].cat.add_categories(["Standard"])
    )
    df_completo["livello_fedelta"].fillna("Standard")

# 2. Imputazione numerica(sconto = 0 se manca la promozione)
df_completo["sconto_categoria_pct"].fillna(0.0)

# 3. Imputazione dell'età con la mediana del gruppo o la mediana generale
mediana_eta = df_completo["eta"].median()
df_completo["eta"].fillna(mediana_eta)

# 4. Rimozione delle righe duplicate
df_completo = df_completo.drop_duplicates()

# 5. Pulizia stringhe (es. rimozione spazi bianchi inizio/fine)
for col in df_completo.select_dtypes(include = (["object", "string"])).columns:
    df_completo[col] = df_completo[col].astype(str).str.strip()

# ==============================================================================
# 5. CALCOLO METRICHE E TRASFORMAZIONE
# ==============================================================================
# Calcoliamo l'importo lordo e l'importo netto applicando lo sconto della categoria
df_completo["totale_lordo"] = (
    df_completo["quantita"] * df_completo["prezzo_listino"]
)
df_completo["totale_netto"] = df_completo["totale_lordo"] * (
    1 - df_completo["sconto_categoria_pct"] / 100
)

print("\n--- Anteprima Dataset Finale Integrato ---")
colonne_visibili = [
    "id_transazione",
    "nome_cliente",
    "livello_fedelta",
    "codice_prodotto",
    "categoria",
    "quantita",
    "totale_netto",
]
print(df_completo[colonne_visibili].head())

# ==============================================================================
# STEP MAPPING DI TUTTE LE COLONNE CATEGORICHE
# ==============================================================================

# Identifichiamo tutte le colonne testuali/categoriche

colonne_category = df_completo.select_dtypes(
    include=["category"]
).columns

# Dizionario master che conterrà i mapping di TUTTE le colonne
# Struttura finale: { 'nome_colonna': {0: 'ValoreA', 1: 'ValoreB', ...} }
dizionario_mapping_completo = {}

for col in colonne_category:

    # Estraiamo il mapping {Codice: Categoria} per questa colonna
    mapping_colonna = dict(enumerate(df_completo[col].cat.categories))

    # Salviamo il mapping nel dizionario principale
    dizionario_mapping_completo[col] = mapping_colonna

    # Applichiamo i codici numerici alla colonna del DataFrame
    df_completo[col] = df_completo[col].cat.codes

# ==============================================================================
# STEP SALVATAGGIO DATASET PULITO (clean_data.csv)
# ==============================================================================

percoso_file_completo = cartella_script /"clean_data.csv"

df_completo.to_csv(percoso_file_completo, index=False, encoding="utf-8")
print("\n[OK] Dataset pulito salvato in 'clean_data.csv'")

# ==============================================================================
# STEP SALVATAGGIO MAPPING COMPLETO (mapping_categorie.pkl)
# ==============================================================================

# Salviamo il dizionario con tutti i mapping in un unico file binario Pickle
with open(cartella_script/"mapping_categories.pkl","wb") as file:
    pickle.dump(dizionario_mapping_completo, file)

print("[OK] Mapping di tutte le categorie salvato in 'mapping_categorie.pkl'")

# ==============================================================================
# STEP 6: VERIFICA E LETTURA DEL FILE PICKLE
# ==============================================================================

# Esempio di come ricaricare e consultare i mapping salvati
with open(cartella_script/"mapping_categories.pkl","rb") as file:
    mapping_ricaricato = pickle.load(file)

print("\n--- Struttura del Mapping Ricaricato ---")
for col_name, mapping in mapping_ricaricato.items():
    print(f"\nColonna '{col_name}':")
    print(mapping)


print("\n--- ISPEZIONE FINALE DEL DATAFRAME ---")
print(df_completo.info(memory_usage="deep"))



File vendite_grandi.csv creato.!
Database catalogo.db creato con successo.!
--- 1. Lettura e Appiattimento JSON Clienti ---
Clienti caricati: 100 righe
  id_cliente      nome_cliente livello_fedelta citta_residenza
0   CLI_0001  Cliente_CLI_0001        Platinum          Milano
1   CLI_0002  Cliente_CLI_0002        Platinum            Roma
2   CLI_0003  Cliente_CLI_0003          Silver         Bologna

--- 2. Lettura CSV Vendite ---
Transazioni caricate: 500 righe.
   id_transazione id_cliente codice_prodotto  quantita            data_ora
0            1000   CLI_0052          PROD_D         4 2025-01-01 00:00:00
1            1001   CLI_0093          PROD_B         4 2025-01-01 00:30:00
2            1002   CLI_0015          PROD_D         5 2025-01-01 01:00:00

--- 3. Lettura DB SQL Catalogo ---
Prodotti nel catalogo: 4 righe.
  codice_prodotto    categoria  prezzo_listino  sconto_categoria_pct
0          PROD_A  Elettronica           120.0                   5.0
1          PROD_B  Inform